# SegOnWeb backend for SegRef3D

## Before running: select a GPU runtime

In Colab, select **Runtime > Change runtime type > T4 GPU > Save**. Another assigned NVIDIA GPU can also be used.

1. In SegRef3D Lite Web, open **Seg on Web > AI Tracking Setup**.
2. Set each object's tracking range and add one or more box keyframes, then choose **Create Input ZIP**.
3. In this notebook, choose **Runtime > Run all**.
4. Upload the generated `segonweb_input.zip` when prompted.
5. The final cell automatically starts downloading `segref3d_result.zip`.
6. In SegRef3D Lite Web, choose **Seg on Web > Import AI Result**.

There is no Gradio screen in this workflow. All keyframe boxes for each object are validated from the ZIP and submitted to the same SAM2 tracking state in both directions. Legacy single-prompt ZIPs remain supported.


In [ ]:
from pathlib import Path
from google.colab import files

print('Upload the segonweb_input.zip exported by SegRef3D.')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError('Upload exactly one SegOnWeb input ZIP, then run all cells again.')
input_path = Path('/content/segonweb_input.zip')
input_path.write_bytes(uploaded[zip_names[0]])
print('Input ready:', input_path.name)


In [ ]:
# Keep the PyTorch installation used by the working v4.8 reference notebook.
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [ ]:
import os
import sys
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('cuDNN version:', torch.backends.cudnn.version())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('A Colab GPU runtime is required. Select Runtime > Change runtime type > T4 GPU.')


In [ ]:
# Preserve the SAM2.1 package commit, checkpoint, and model config from v4.8.
!{sys.executable} -m pip install opencv-python matplotlib tqdm
!{sys.executable} -m pip install --no-build-isolation 'git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4'
!mkdir -p /content/checkpoints
!wget -q -O /content/checkpoints/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
!wget -q -O /content/segmentation_job.py https://raw.githubusercontent.com/SatoruMuro/SegRef3D/main/SegRef3D/segmentation_job.py
!wget -q -O /content/segonweb_backend.py https://raw.githubusercontent.com/SatoruMuro/SegRef3D/main/ColabNotebooks/segonweb_backend.py


In [ ]:
# Predictor initialization intentionally matches SAM2GUIforImgSeqv4_8.ipynb.
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
device = torch.device('cuda')
torch.autocast('cuda', dtype=torch.float16).__enter__()
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

from sam2.build_sam import build_sam2_video_predictor

sam2_checkpoint = '/content/checkpoints/sam2.1_hiera_large.pt'
model_cfg = 'configs/sam2.1/sam2.1_hiera_l.yaml'
try:
    predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
except Exception as exc:
    raise RuntimeError(f'SAM2 model loading failed. Check the GPU runtime and rerun all cells: {exc}') from exc
print('SAM2.1 predictor ready on:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from html import escape
import numpy as np
from IPython.display import Audio, HTML, display
from tqdm.auto import tqdm

from segonweb_backend import SegOnWebProcessingError, process_segmentation_job

class NotebookProgress:
    def __init__(self):
        self.bar = None

    def __call__(self, info):
        event = info['event']
        if event == 'step':
            print(f"Step {info['step']}/{info['total_steps']}: {info['message']}")
        elif event == 'start':
            print(f"Objects: {info['object_count']} | Frames: {info['frame_count']}")
            self.bar = tqdm(total=info['total_work'], unit='frame', desc='Segmenting')
        elif event == 'object':
            obj = info['object']
            message = f"Object {info['object_position']}/{info['object_count']}: {obj['name']} (ID {obj['id']})"
            print(message)
            if self.bar is not None:
                self.bar.set_description(message)
        elif event == 'prompt':
            print(f"  {info['direction'].title()} prompt {info['prompt_position']}/{info['prompt_count']}: frame {info['frame'] + 1}")
            if self.bar is not None:
                self.bar.set_postfix(direction=info['direction'], prompt=info['prompt_position'], frame=info['frame'] + 1)
        elif event == 'frame' and self.bar is not None:
            self.bar.update(1)
            self.bar.set_postfix(direction=info['direction'], frame=info['frame'] + 1)
        elif event == 'complete' and self.bar is not None:
            self.bar.n = self.bar.total
            self.bar.refresh()
            self.bar.close()

result_path = None

result_path = Path('/content/segref3d_result.zip')
try:
    process_segmentation_job(
        str(input_path),
        predictor,
        work_dir='/content/segonweb_work',
        output_zip=str(result_path),
        device_name=torch.cuda.get_device_name(0),
        progress_callback=NotebookProgress(),
    )
except SegOnWebProcessingError as exc:
    result_path = None
    display(HTML(f'<h3 style="color:#b00020">SegOnWeb failed</h3><p>{escape(str(exc))}</p>'))
except Exception as exc:
    result_path = None
    display(HTML(f'<h3 style="color:#b00020">Unexpected processing failure</h3><p>{escape(str(exc))}</p>'))

if result_path is not None and result_path.exists():
    display(HTML('<h2 style="color:#137333">Segmentation complete</h2>'))
    sample_rate = 22050
    t = np.linspace(0, 0.18, int(sample_rate * 0.18), endpoint=False)
    chime = np.concatenate([0.25 * np.sin(2 * np.pi * 880 * t), np.zeros(1200), 0.25 * np.sin(2 * np.pi * 660 * t)])
    display(Audio(chime, rate=sample_rate, autoplay=True))


In [ ]:
from google.colab import files

if result_path is None or not result_path.exists():
    raise RuntimeError('Result ZIP was not generated. Review the processing cell output before retrying.')

print('Starting download:', result_path.name)
files.download(str(result_path))


## Reference compatibility

The SAM2 execution path above retains the working v4.8 reference implementation's pinned SAM2 commit, SAM2.1 Hiera Large checkpoint/config, CUDA autocast/TF32 setup, video predictor initialization, `add_new_points_or_box` box format, forward propagation, reversed-frame backward propagation, `logits > 0.0` mask threshold, and uint8 single-label PNG output. Only the Gradio and input/output orchestration has been replaced.


Copyright (c) 2024 Satoru Muro. All rights reserved.

Redistribution and use in source and binary forms, with or without modification, are permitted under the license terms distributed with the SegRef3D repository.
